# populariteit bepalen van tedtalk videos

# eerst de data ophalen en opschonen

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import joblib
from _datetime import datetime
import re

df = pd.read_csv("Kaggle_TED_video_metadata_balanced.csv")
# df = pd.read_csv("goodKaggle - Kaggle.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          600 non-null    object
 1   tags           590 non-null    object
 2   views          600 non-null    int64 
 3   likes          600 non-null    int64 
 4   dislikes       600 non-null    int64 
 5   comment_count  600 non-null    int64 
 6   published_at   600 non-null    object
 7   duration       600 non-null    object
 8   category_id    600 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 42.3+ KB


,views,likes,dislikes,comment_count,category_id
count,6.000000e+02,6.000000e+02,600.0,600.000000,600.000000
mean,1.066151e+06,2.263473e+04,0.0,1070.870000,25.030000
std,3.611008e+06,1.000825e+05,0.0,4055.503954,4.812244
min,1.000000e+00,0.000000e+00,0.0,0.000000,1.000000
25%,5.922150e+04,6.242500e+02,0.0,86.000000,22.000000
50%,1.332595e+05,1.846000e+03,0.0,202.000000,27.000000
75%,4.101668e+05,6.382250e+03,0.0,511.750000,28.000000
max,5.562224e+07,1.921445e+06,0.0,77980.000000,29.000000


##### met de column title kan het model niet veel. Daarom maak ik er een column van met de hoeveelheid karakters in de titel.

In [28]:
df["title_length"] = df['title'].apply(len)
df.head(5)

,title,tags,views,likes,dislikes,comment_count,published_at,duration,category_id,title_length
0,Stories from a home for terminally ill childre...,"TED Talk,TED Talks,Children,Community,Death,Fa...",77455,1768,0,49,2017-03-24T15:32:48Z,PT15M19S,29,60
1,Why our screens make us less happy | Adam Alter,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",800326,20579,0,572,2017-08-01T15:29:04Z,PT9M30S,22,47
2,A tribute to nurses | Carolyn Jones,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",87635,1877,0,48,2017-05-30T18:17:56Z,PT10M49S,22,35
3,"Asking for help is a strength, not a weakness ...","TEDTalk,TEDTalks,Children,Communication,Commun...",190840,4726,0,187,2017-04-12T15:17:51Z,PT11M56S,22,67
4,Don't feel sorry for refugees -- believe in th...,"TEDTalk,TEDTalks,Children,Global issues,Humani...",98523,2669,0,226,2017-07-25T15:06:25Z,PT14M14S,29,62


##### de title is nutteloos voor machine learning, dus die halen we weg. Ook zijn de dislikes allemaal 0, omdat youtube dit uitgeschakeld heeft. Daarom verwijder ik deze column ook.

In [29]:
df = df.drop(columns=["title", "dislikes"], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tags           590 non-null    object
 1   views          600 non-null    int64 
 2   likes          600 non-null    int64 
 3   comment_count  600 non-null    int64 
 4   published_at   600 non-null    object
 5   duration       600 non-null    object
 6   category_id    600 non-null    int64 
 7   title_length   600 non-null    int64 
dtypes: int64(5), object(3)
memory usage: 37.6+ KB


##### de column "tags" bevat 10 null waarden. Hier ga ik echter niks aan doen, omdat dat natuurlijk voorkomt in de dataset. Er worden in de realiteit video's geopload zonder tags. Verder zijn er geen null waarden.

##### De tag column is nog niet leesbaar voor machine learning. Als ik one-hot encoding zou toepassen, heb ik 591 comlumns. Dit is niet handig omdat ik dan weer andere tags heb voor mijn eigen dataset. Daarom maak ik 1 column met de hoeveelheid tags.

In [30]:
df['num_tags'] = df['tags'].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)
df = df.drop("tags", axis=1)
df.head(5)

,views,likes,comment_count,published_at,duration,category_id,title_length,num_tags
0,77455,1768,49,2017-03-24T15:32:48Z,PT15M19S,29,60,18
1,800326,20579,572,2017-08-01T15:29:04Z,PT9M30S,22,47,9
2,87635,1877,48,2017-05-30T18:17:56Z,PT10M49S,22,35,15
3,190840,4726,187,2017-04-12T15:17:51Z,PT11M56S,22,67,12
4,98523,2669,226,2017-07-25T15:06:25Z,PT14M14S,29,62,9


##### De published_at column is niet bruikbaar voor machine learning, daarom pas ik er wat feature engineering op toe. Eerst de dag, maand, jaar en uur scheiden.

In [31]:
df['published_at'] = pd.to_datetime(df['published_at'])

df['year_published'] = df['published_at'].dt.year
df['month_published'] = df['published_at'].dt.month
df['day_published'] = df['published_at'].dt.day
df['hour_published'] = df['published_at'].dt.hour

##### Ook kan ik de dag van de week opslaan, en de tijd sinds upload in een aparte column stoppen. de column published at hebben we niet meer nodig.

In [32]:
df['published_at'] = df['published_at'].dt.tz_localize(None)
df['days_since_published'] = (datetime.now() - df['published_at']).dt.days

df = df.drop("published_at", axis= 1)
df.head(5)

,views,likes,comment_count,duration,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published
0,77455,1768,49,PT15M19S,29,60,18,2017,3,24,15,2761
1,800326,20579,572,PT9M30S,22,47,9,2017,8,1,15,2631
2,87635,1877,48,PT10M49S,22,35,15,2017,5,30,18,2694
3,190840,4726,187,PT11M56S,22,67,12,2017,4,12,15,2742
4,98523,2669,226,PT14M14S,29,62,9,2017,7,25,15,2638


#### met de duration kan ik niet veel in deze format. Daarom ga ik het omzetten naar een leesbaar format

In [33]:
df['duration_seconds'] = df['duration'].apply(lambda x: (int(re.match(r'PT(\d+)M', x).group(1)) * 60 if isinstance(x, str) and re.match(r'PT(\d+)M', x) else 0) + 
                                                            (int(re.match(r'PT(\d+)S', x).group(1)) if isinstance(x, str) and re.match(r'PT(\d+)S', x) else 0))
df = df.drop("duration", axis=1)
df.head(3)

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds
0,77455,1768,49,29,60,18,2017,3,24,15,2761,900
1,800326,20579,572,22,47,9,2017,8,1,15,2631,540
2,87635,1877,48,22,35,15,2017,5,30,18,2694,600


# engagement rate, views over tijd, gemiddelde weergaven van category

In [34]:
df['engagement_rate'] = df['likes'] / df['views']

df['views_over_time'] = df['views'] / df['days_since_published']
df.head()


,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,77455,1768,49,29,60,18,2017,3,24,15,2761,900,0.022826,28.053242
1,800326,20579,572,22,47,9,2017,8,1,15,2631,540,0.025713,304.190802
2,87635,1877,48,22,35,15,2017,5,30,18,2694,600,0.021418,32.529696
3,190840,4726,187,22,67,12,2017,4,12,15,2742,660,0.024764,69.598833
4,98523,2669,226,29,62,9,2017,7,25,15,2638,840,0.027090,37.347612


### omdat ik later een scaler nodig heb met de later bepaalde relevante columns, moet ik die fitten en opslaan (ik krijg een error wanneer ik wil scalen en columns mis)

In [35]:
relevant_scaler = StandardScaler()
relevant_columns = ["views", "likes", "comment_count", "engagement_rate","views_over_time"]
relevant_scaler.fit(df[relevant_columns])

StandardScaler()

# Alles schalen

In [36]:
scaler = StandardScaler()
columns_to_scale = ["views_over_time", "engagement_rate", "views", "likes", "comment_count", "title_length", "num_tags", "year_published", "month_published", "hour_published", "days_since_published", "duration_seconds", "day_published"]
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
df.head(5)

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,-0.274029,-0.208669,-0.252181,29,0.489379,0.735176,1.242757,-0.996458,1.026353,-0.410958,-1.166477,0.277560,0.926415,-0.148701
1,-0.073677,-0.020557,-0.123113,22,-0.435096,-0.524726,1.242757,0.544457,-1.590250,-0.410958,-1.270751,-0.677530,1.263133,-0.024974
2,-0.271208,-0.207579,-0.252428,22,-1.288457,0.315209,1.242757,-0.380092,1.708945,0.512544,-1.220219,-0.518349,0.762229,-0.146695
3,-0.242603,-0.179089,-0.218125,22,0.987173,-0.104759,1.242757,-0.688275,-0.338831,-0.410958,-1.181717,-0.359167,1.152445,-0.130086
4,-0.268190,-0.199659,-0.208501,29,0.631606,-0.524726,1.242757,0.236274,1.140118,-0.410958,-1.265137,0.118378,1.423712,-0.144536


# Dataframe opslaan en schaler opslaan voor hergebruik

In [37]:
df.to_csv("./clean_kaggle_data.csv")

scaler_filename = "scaler.save"
joblib.dump(relevant_scaler, scaler_filename) 

['scaler.save']

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   views                 600 non-null    float64
 1   likes                 600 non-null    float64
 2   comment_count         600 non-null    float64
 3   category_id           600 non-null    int64  
 4   title_length          600 non-null    float64
 5   num_tags              600 non-null    float64
 6   year_published        600 non-null    float64
 7   month_published       600 non-null    float64
 8   day_published         600 non-null    float64
 9   hour_published        600 non-null    float64
 10  days_since_published  600 non-null    float64
 11  duration_seconds      600 non-null    float64
 12  engagement_rate       600 non-null    float64
 13  views_over_time       600 non-null    float64
dtypes: float64(13), int64(1)
memory usage: 65.8 KB


In [39]:
df.head()

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,-0.274029,-0.208669,-0.252181,29,0.489379,0.735176,1.242757,-0.996458,1.026353,-0.410958,-1.166477,0.277560,0.926415,-0.148701
1,-0.073677,-0.020557,-0.123113,22,-0.435096,-0.524726,1.242757,0.544457,-1.590250,-0.410958,-1.270751,-0.677530,1.263133,-0.024974
2,-0.271208,-0.207579,-0.252428,22,-1.288457,0.315209,1.242757,-0.380092,1.708945,0.512544,-1.220219,-0.518349,0.762229,-0.146695
3,-0.242603,-0.179089,-0.218125,22,0.987173,-0.104759,1.242757,-0.688275,-0.338831,-0.410958,-1.181717,-0.359167,1.152445,-0.130086
4,-0.268190,-0.199659,-0.208501,29,0.631606,-0.524726,1.242757,0.236274,1.140118,-0.410958,-1.265137,0.118378,1.423712,-0.144536
